# GFS Atmospheric Data Downloader
## Retrieving Wind, Temperature, and Pressure Forecasts from NOAA's Global Forecast System

This notebook provides a self-contained utility for downloading sub-regional atmospheric forecast data from NOAA's **Global Forecast System (GFS)** via the UCAR THREDDS Data Server (TDS) Network Common Subsetted Service (NCSS).

The downloader fetches four forecast variables at 100 m hub height for a user-specified country bounding box, across all four daily GFS model initialisation runs (00Z, 06Z, 12Z, 18Z), and saves the output as NetCDF files for downstream wind-energy analysis.

**Outputs consumed by:** `wind_power_density_analysis.ipynb`

## 1. Dependencies

Install the required packages with:

```bash
pip install siphon xarray netCDF4 tqdm pandas
```

| Package | Role |
|---------|------|
| `siphon` | THREDDS catalog access and NCSS queries |
| `xarray` | In-memory NetCDF representation |
| `netCDF4` | Low-level backend for `xarray` |
| `tqdm` | Progress bar for the four-run loop |
| `pandas` | Timestamp utilities |

## 2. Country Bounding-Box Configuration

The downloader reads bounding boxes from a JSON file at `countries/country_bbox.json`.  
The expected schema is:

```json
{
  "EU": ["Europe", [270.0, 24.5, 45.0, 84.75]],
  "DE": ["Germany", [5.5,  47.0, 15.5, 55.5]],
  "FR": ["France",  [-5.5, 41.0, 10.0, 51.5]]
}
```

Each entry maps an **ISO-style country code** to a two-element list:

1. `country_name` (string) — used as part of the output filename.
2. `bbox` (list of four floats) — `[west_lon, south_lat, east_lon, north_lat]` in degrees.  
   Longitudes for the GFS global grid must be in the 0–360° range used by NCSS.

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import json
import logging

from siphon.catalog import TDSCatalog
import pandas as pd
import xarray as xr
import netCDF4  # noqa: F401  — required as xarray NetCDF backend
from tqdm import tqdm

## 3. Logging and Path Configuration

Configure the logging level so that progress messages and errors from the downloader are visible in the notebook output.

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Paths — edit these to match your local setup
JSON_PATH  = "countries/country_bbox.json"
OUTPUT_DIR = "gfs_data"
RUN_HOURS  = [0, 6, 12, 18]  # all four daily GFS initialisation times (UTC)
FORECAST_DAYS = 3

## 4. Download Functions

### `get_country_info`
Reads the bounding-box JSON and returns the country name and spatial extent for a given country code.

### `data_downloader`
Main entry point.  For each requested model-run hour it:

1. Constructs a time range from that run's initialisation time to `days` days ahead.
2. Queries the NCSS endpoint for U-wind, V-wind, temperature, and pressure on the bounding box.
3. Saves the result as a NetCDF file named  
   `gfs_<CountryName>_<YYYYMMDD>_run<HH>Z_<N>day.nc`.

In [3]:
# ── helper ──────────────────────────────────────────────────────────────────
def get_country_info(country_code: str, json_file_path: str):
    try:
        json_file_path = Path(json_file_path)
        with open(json_file_path, 'r') as file:
            countries_data = json.load(file)
        if country_code in countries_data:
            country_name, bbox = countries_data[country_code]
            country_name = country_name.replace(" ", "_")
            return country_name, bbox
        else:
            logging.error(f"Country code {country_code} not found in data.")
            return None
    except Exception as e:
        logging.error(f"Error reading JSON file: {e}")
        return None


# ── main downloader ──────────────────────────────────────────────────────────
def data_downloader(
        country_code: str,
        json_file_path: str,
        days: int = 3,
        run_hours: list[int] = None,
        output_dir: str = 'gfs_data'
):
    """
    Download and save GFS data for a specified country and time range.

    Parameters
    ----------
    country_code : str
        ISO-style country code matching a key in the bounding-box JSON.
    json_file_path : str
        Path to the country bounding-box JSON file.
    days : int, optional
        Forecast horizon in days (default 3 → 72 h).
    run_hours : list[int], optional
        GFS model runs to download (UTC hours).  Any subset of {0, 6, 12, 18}.
        Defaults to all four runs.
    output_dir : str, optional
        Directory for output NetCDF files (created if absent).
    """
    if run_hours is None:
        run_hours = [0, 6, 12, 18]
    if not all(h in {0, 6, 12, 18} for h in run_hours):
        raise ValueError("run_hours must be a subset of [0, 6, 12, 18]")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    country_info = get_country_info(country_code, json_file_path)
    if not country_info:
        logging.error(f"Could not retrieve country info for {country_code}")
        return
    country_name, (west, south, east, north) = country_info

    catalog_url = (
        "https://thredds.ucar.edu/thredds/catalog/grib/NCEP/GFS/"
        "Global_0p25deg/catalog.xml"
    )
    try:
        best_gfs = TDSCatalog(catalog_url)
    except Exception as e:
        logging.error(f"Failed to open catalog: {e}")
        return

    for run_h in tqdm(run_hours, desc="Model runs"):
        # Use timezone-aware UTC datetime (datetime.utcnow is deprecated in Python 3.12+)
        run_time = datetime.now(timezone.utc).replace(
            hour=run_h, minute=0, second=0, microsecond=0
        )
        forecast_end = run_time + timedelta(days=days)

        ncss = best_gfs.datasets[
            'Latest Collection for GFS Quarter Degree Forecast'
        ].subset()
        query = ncss.query()
        query.variables(
            'u-component_of_wind_height_above_ground',
            'v-component_of_wind_height_above_ground',
            'Temperature_height_above_ground',
            'Pressure_height_above_ground'
        )
        query.add_lonlat()
        query.time_range(run_time, forecast_end)
        query.lonlat_box(north=north, south=south, east=east, west=west)
        query.accept('netcdf')

        try:
            data = ncss.get_data(query)
            ds = xr.open_dataset(xr.backends.NetCDF4DataStore(data))

            fname = (
                f"gfs_{country_name}_{run_time:%Y%m%d}"
                f"_run{run_h:02d}Z_{days}day.nc"
            )
            out_file = output_dir / fname
            ds.to_netcdf(out_file)
            logging.info(f"Saved {out_file}")
        except Exception as e:
            logging.error(f"Failed to download run {run_h:02d}Z: {e}")

## 5. Example Usage

The cell below downloads a 3-day GFS forecast for Europe across all four model runs.  
Running this cell requires an active internet connection and takes approximately 7–10 minutes.

The four output files are written to `gfs_data/` and named:
```
gfs_Europe_YYYYMMDD_run00Z_3day.nc
gfs_Europe_YYYYMMDD_run06Z_3day.nc
gfs_Europe_YYYYMMDD_run12Z_3day.nc
gfs_Europe_YYYYMMDD_run18Z_3day.nc
```

In [4]:
data_downloader(
    country_code="EU",
    json_file_path=JSON_PATH,
    days=FORECAST_DAYS,
    run_hours=RUN_HOURS,
    output_dir=OUTPUT_DIR
)

Model runs: 100%|██████████| 4/4 [06:54<00:00, 103.67s/it]
